In [ ]:
# Network segments to evaluate. Edit these before running.
# Enter any address in the target /24 (e.g. "10.1.5.0", "10.1.5.0/24", or "10.1.5").
# Only the first three octets are used.
oco = "192.168.10.0"   # OCO: the subnet YOUR group attacked              (matched on dest_ip)
dco = "192.168.20.0"   # DCO: YOUR group's subnet, attacked by another group (matched on dest_ip)


In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Wed May 18 02:56:50 2022

@author: Tom
"""

import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler
import traceback
import sys
import datetime
import matplotlib.pyplot as plt

from df_binning import *
from generate_individual_labels_dataframe import *

#############
# Run variables
#############

benign_label = "none"

conn_server_loc = '/home/kali/datasets/CNT4416_full_csv'

# Progress log written next to the notebook (the original path was in Tom's home directory)
log_location = 'rf_log_' + str(datetime.date.today()) + '.txt'

########################
# functions
########################

def segment_prefix(addr):
    """First three octets plus a trailing dot, e.g. '10.1.5.0/24' -> '10.1.5.'"""
    octets = addr.strip().split("/")[0].split(".")
    if len(octets) < 3 or not all(o.isdigit() and 0 <= int(o) <= 255 for o in octets[:3]):
        raise ValueError("'" + addr + "' is not a valid IPv4 network segment")
    return ".".join(octets[:3]) + "."

def segment_confusion_matrix(preds, ip_col, prefix):
    """2x2 matrix (rows = actual, cols = predicted; order 0.0, 1.0) counting only the
    rows whose ip_col falls in the /24. Always 2x2, even if one class is absent."""
    subset = preds.filter(col(ip_col).cast("string").startswith(prefix))
    counts = {(r["label_bin"], r["prediction"]): r["count"]
              for r in subset.groupBy("label_bin", "prediction").count().collect()}
    vals = [0.0, 1.0]
    return np.array([[counts.get((a, p), 0) for p in vals] for a in vals], dtype=int)

oco_prefix = segment_prefix(oco)
dco_prefix = segment_prefix(dco)
print("OCO segment (dest_ip): " + oco_prefix + "x")
print("DCO segment (dest_ip): " + dco_prefix + "x")

########################
# code start
########################

spark = ( 
    SparkSession
    .builder
    .master("spark://hadoop-master:7077")
    .appName("TM_RandoForest")
    .config("spark.driver.cores", '4')
    .config("spark.driver.memory", "12g")
    .config("spark.executor.cores",'20')
    .config("spark.executor.memory", "40g")
    .config("spark.sql.shuffle.partitions",'240')
    .config("spark.submit.deployMode", "client")
    .getOrCreate()
    )

printToLog("Begin run\n-----------\n----------\n\n", log_location)
printToLog("Dataset location - " + conn_server_loc, log_location)

source_df = spark.read.option("header", True).options(inferSchema=True).csv(conn_server_loc)
 
try:
    source_df = rename_columns(source_df)
except Exception as e:
    print("Error:  " + str(e), file=sys.stderr)
    print(traceback.format_exc())   

# Binary label: any technique label other than "none" = attack (0.0), "none" = benign (1.0)
unbinned_df = source_df.withColumn("label_bin", when((col("label_multi") != benign_label), 0.0).otherwise(1.0))
printToLog(str(unbinned_df.count()) + " rows", log_location)

#Carry out the binning on the df
numeric_percent_trim = 0.02
nominal_percent_agg = 0.98
replace_bool = False

attrList_ip_addr = ['dest_ip','src_ip']
attrList_port = ['dest_port', 'src_port']
attrList_bool = ['local_orig', 'local_resp']
attrList_nominal = ['protocol', 'conn_state', 'history','service']
attrList_numeric = ['duration','orig_bytes','orig_pkts','orig_ip_bytes','resp_bytes','resp_pkts','resp_ip_bytes','missed_bytes']

printToLog("Binning start", log_location)
conn_df = genFullBinnedDF(unbinned_df, attrList_ip_addr, attrList_port, attrList_bool, attrList_nominal, nominal_percent_agg, attrList_numeric, numeric_percent_trim, replace_bool).persist()
printToLog("Binning finished", log_location)

########################
# Build Feature cols
########################     

feature_cols = []
for clm, types in conn_df.dtypes:
    if "_bin" in clm:
        if not "label" in clm:
            feature_cols.append(clm)

# build feature vector for Classifier
assembler = VectorAssembler(inputCols = feature_cols, outputCol = "features")
conn_df = assembler.transform(conn_df)

printToLog("feature_cols - " + str(feature_cols), log_location)
print(str(len(feature_cols)) + " features: " + str(feature_cols))

########################
# Model: trained on the entire dataset
########################   

rf = RandomForestClassifier(featuresCol = "features", labelCol = "label_bin")
rfModel = rf.fit(conn_df)
printToLog("randomForest model fit", log_location)

#############
# Predictions on the entire dataset
#############   

predictions = rfModel.transform(conn_df)

for ip_c in ("dest_ip",):
    if ip_c not in predictions.columns:
        raise RuntimeError("'" + ip_c + "' is not in the binned dataframe; segment filtering needs it")

################################
# Output: confusion matrices counting only connections destined to the OCO and DCO subnets
################################

class_names = ["attack", "benign"]   # label_bin: attack = 0.0, benign = 1.0
segments = [("OCO", "dest_ip", oco_prefix), ("DCO", "dest_ip", dco_prefix)]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, (seg_name, ip_col, prefix) in zip(axes, segments):
    cm = segment_confusion_matrix(predictions, ip_col, prefix)
    n = int(cm.sum())

    print("\n" + seg_name + "  (" + ip_col + " " + prefix + "x, " + "{:,}".format(n) + " connections)")
    print("                   predicted attack   predicted benign")
    print("  actual attack    %16d   %16d" % (cm[0, 0], cm[0, 1]))
    print("  actual benign    %16d   %16d" % (cm[1, 0], cm[1, 1]))
    if n == 0:
        print("  No connections matched " + prefix + "x - check the " + seg_name + " address.")

    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, "{:,}".format(cm[i, j]), ha="center", va="center", fontsize=14,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["predicted " + c for c in class_names])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["actual " + c for c in class_names])
    ax.set_title(seg_name, fontsize=18, fontweight="bold", pad=28)
    ax.text(0.5, 1.02, ip_col + " " + prefix + "x  |  " + "{:,}".format(n) + " connections",
            transform=ax.transAxes, ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

conn_df.unpersist()
printToLog("End run\n-----------\n----------\n\n", log_location)
